# E-commerce Olist — EDA + Mineração (dados reais)
Objetivo: segmentar clientes (RFM + KMeans k=4), prever satisfação (árvore + matriz de confusão) e extrair associação (co-ocorrência de categorias) para logística e marketing.

> **Dados reais — Olist Brazilian E-commerce** (`olistbr/brazilian-ecommerce`, Kaggle): 9 CSVs em `data/` (gitignored, nunca commitados) — **99.441 pedidos** (04/set/2016–17/out/2018), **112.650 itens**, **99.441 registros de cliente** (96.096 únicos), 27 UFs, 73 categorias de produto. Join `orders ⨝ items ⨝ customers` (+ reviews e categoria do produto) = **112.650 linhas** (98.666 pedidos com itens). Seed 42 usada só onde há aleatoriedade (`train_test_split`, `KMeans`, amostra publicada).

In [1]:
import sys
from pathlib import Path
ROOT = Path('.').resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent  # executado a partir de notebooks/
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
SAMPLE_CSV = ROOT / 'docs' / 'assets' / 'data' / 'sample_olist.csv'
import itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.common import df_to_html_table, load_csv, save_fig
from src.mining import confusion_summary, run_kmeans, run_tree

## 1. Entendimento — dicionário dos 9 CSVs
Fonte: `kaggle datasets download olistbr/brazilian-ecommerce` (CSVs em `data/`, gitignored). Dicionário + volumes reais (linhas por arquivo) via `df_to_html_table`:

In [2]:
dict9 = pd.DataFrame([
    ("olist_orders_dataset.csv", "pedidos: status, timestamps de compra/aprovacao/entrega"),
    ("olist_order_items_dataset.csv", "itens: preco, frete, vendedor por pedido/produto"),
    ("olist_customers_dataset.csv", "clientes: id, cidade/UF"),
    ("olist_products_dataset.csv", "produtos: categoria, dimensoes, peso"),
    ("olist_order_reviews_dataset.csv", "avaliacoes: review_score 1-5"),
    ("olist_order_payments_dataset.csv", "pagamentos: tipo, parcelas, valor"),
    ("olist_sellers_dataset.csv", "vendedores: cidade/UF"),
    ("olist_geolocation_dataset.csv", "geolocalizacao: CEP/lat-lng por cidade"),
    ("product_category_name_translation.csv", "traducao categoria PT-BR -> EN"),
], columns=["arquivo", "conteudo"])
display(HTML(df_to_html_table(dict9, max_rows=20)))
vol = pd.DataFrame([
    (f, len(load_csv(DATA / f))) for f in [
        "olist_orders_dataset.csv", "olist_order_items_dataset.csv",
        "olist_customers_dataset.csv", "olist_products_dataset.csv",
        "olist_order_reviews_dataset.csv", "olist_order_payments_dataset.csv",
        "olist_sellers_dataset.csv", "olist_geolocation_dataset.csv",
        "product_category_name_translation.csv"]
], columns=["arquivo", "linhas"])
print(f"total de linhas nos 9 CSVs: {vol['linhas'].sum()}")
display(HTML(df_to_html_table(vol, max_rows=20)))

arquivo,conteudo
olist_orders_dataset.csv,"pedidos: status, timestamps de compra/aprovacao/entrega"
olist_order_items_dataset.csv,"itens: preco, frete, vendedor por pedido/produto"
olist_customers_dataset.csv,"clientes: id, cidade/UF"
olist_products_dataset.csv,"produtos: categoria, dimensoes, peso"
olist_order_reviews_dataset.csv,avaliacoes: review_score 1-5
olist_order_payments_dataset.csv,"pagamentos: tipo, parcelas, valor"
olist_sellers_dataset.csv,vendedores: cidade/UF
olist_geolocation_dataset.csv,geolocalizacao: CEP/lat-lng por cidade
product_category_name_translation.csv,traducao categoria PT-BR -> EN


total de linhas nos 9 CSVs: 1550922


arquivo,linhas
olist_orders_dataset.csv,99441
olist_order_items_dataset.csv,112650
olist_customers_dataset.csv,99441
olist_products_dataset.csv,32951
olist_order_reviews_dataset.csv,99224
olist_order_payments_dataset.csv,103886
olist_sellers_dataset.csv,3095
olist_geolocation_dataset.csv,1000163
product_category_name_translation.csv,71


## 2. Preparação — dados reais: join, nulos, outliers
Join `orders ⨝ order_items ⨝ customers` (+ reviews deduplicadas por pedido, + categoria do produto). `delay_days = delivered_customer − estimated_delivery` (dias); nulos (~3%: canceled/unavailable/shipped sem entrega) viram flag `is_null_delivered` e saem do cálculo. Pedidos com >1 avaliação (551) ficam com a mais recente. Outliers de preço/frete (IQR) com clip no p99.

In [3]:
orders = load_csv(DATA / "olist_orders_dataset.csv")
items_raw = load_csv(DATA / "olist_order_items_dataset.csv")
customers = load_csv(DATA / "olist_customers_dataset.csv")
reviews_raw = load_csv(DATA / "olist_order_reviews_dataset.csv")
products = load_csv(DATA / "olist_products_dataset.csv")
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
orders["delay_days"] = (orders["order_delivered_customer_date"]
                        - orders["order_estimated_delivery_date"]).dt.days
print(f"pedidos={len(orders)} itens={len(items_raw)} "
      f"clientes={len(customers)} (unicos={customers['customer_unique_id'].nunique()})")
print("status:", orders["order_status"].value_counts().to_dict())
print("periodo:", orders["order_purchase_timestamp"].min(),
      "->", orders["order_purchase_timestamp"].max())
print(f"sem entrega (delay nulo): {int(orders['delay_days'].isna().sum())} "
      f"({orders['delay_days'].isna().mean():.1%})")
reviews_raw["review_creation_date"] = pd.to_datetime(
    reviews_raw["review_creation_date"], errors="coerce")
reviews = (reviews_raw.sort_values("review_creation_date")
           .drop_duplicates("order_id", keep="last"))  # 551 pedidos com >1 review
print(f"reviews: {len(reviews_raw)} brutas -> {len(reviews)} por pedido; "
      f"notas={reviews['review_score'].value_counts().sort_index().to_dict()}")

pedidos=99441 itens=112650 clientes=99441 (unicos=96096)
status: {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}
periodo: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
sem entrega (delay nulo): 2965 (3.0%)
reviews: 99224 brutas -> 98673 por pedido; notas={1: 11365, 2: 3130, 3: 8134, 4: 19043, 5: 57001}


In [4]:
df = (orders.merge(customers, on="customer_id", how="left")
            .merge(items_raw, on="order_id", how="inner")
            .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
            .merge(products[["product_id", "product_category_name"]],
                   on="product_id", how="left"))
print(f"join: {len(df)} linhas, {df['order_id'].nunique()} pedidos, "
      f"{df['customer_unique_id'].nunique()} clientes unicos, "
      f"{df['product_category_name'].nunique()} categorias")
print(f"cobertura review: {df['review_score'].notna().mean():.1%} | "
      f"categoria ausente: {df['product_category_name'].isna().mean():.2%}")
df["is_null_delivered"] = df["order_delivered_customer_date"].isna()
for col in ["price", "freight_value"]:  # outliers IQR -> clip p99
    q1, q3, q99 = df[col].quantile(0.25), df[col].quantile(0.75), df[col].quantile(0.99)
    n_out = int((df[col] > q1 + 1.5 * (q3 - q1)).sum())
    print(f"{col}: {n_out} outliers IQR, clip p99={q99:.2f}")
    df[col] = df[col].clip(upper=q99)
df["freight_ratio"] = df["freight_value"] / df["price"].replace(0, np.nan)
df["is_late"] = (df["delay_days"] > 0).fillna(False).astype(int)
display(HTML(df_to_html_table(df[["order_id", "customer_state", "product_category_name",
    "price", "freight_value", "delay_days", "review_score"]].head(10), max_rows=10)))

join: 112650 linhas, 98666 pedidos, 95420 clientes unicos, 73 categorias
cobertura review: 99.2% | categoria ausente: 1.42%
price: 16640 outliers IQR, clip p99=890.00
freight_value: 19137 outliers IQR, clip p99=84.52


order_id,customer_state,product_category_name,price,freight_value,delay_days,review_score
e481f51cbdc54678b7cc49136f2d6af7,SP,utilidades_domesticas,29.99,8.72,-8.0,4.0
53cdb2fc8bc7dce0b6741e2150273451,BA,perfumaria,118.70,22.76,-6.0,4.0
47770eb9100c2d0c44946d9cf07ec65d,GO,automotivo,159.90,19.22,-18.0,5.0
949d5b44dbf5de918fe9c16f97b45f8a,RN,pet_shop,45.00,27.20,-13.0,5.0
ad21c59c0840e6cb83a9ceb5573f8159,SP,papelaria,19.90,8.72,-10.0,5.0
a4591c265e18cb1dcee52889e2d8acc3,PR,automotivo,147.90,27.36,-6.0,4.0
136cce7faa42fdb2cefd53fdc79a6098,RS,NaN,49.90,16.05,NaN,2.0
6514b8ad8028c9f2cc2374ded245783f,RJ,automotivo,59.99,15.17,-12.0,5.0
76c6e866289321a7c93b82b54852dc33,RS,moveis_decoracao,19.90,16.05,-32.0,1.0
e69bfb5eb88e0ed6a785585b27e16dbf,SP,moveis_escritorio,149.99,19.77,-7.0,5.0


## 3. EDA — atraso × review, receita por categoria/UF, frete por UF, outliers (7 figs `olist_*` via `save_fig`)
Sem distância de CD nos dados reais: `olist_frete_distancia.png` traz o substituto honesto — frete médio por UF (Norte/Nordeste pagam ~2–3× o Sudeste).

In [5]:
g = df.dropna(subset=['review_score']).groupby('review_score')['delay_days'].mean()
print(g.round(2).to_dict())
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(g.index.astype(int).astype(str), g.values, color="#1a5fb4")
ax.set(xlabel="review_score", ylabel="atraso medio (dias)", title="Atraso medio por nota (dados reais)")
fig.tight_layout(); save_fig(fig, 'olist_atraso_review.png'); plt.close(fig)
gcat = df.groupby('product_category_name')['price'].sum().sort_values(ascending=True).tail(10)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(gcat.index, gcat.values, color="#2ca02c")
ax.set(xlabel="receita (R$)", title="Receita por categoria (top 10, dados reais)")
fig.tight_layout(); save_fig(fig, 'olist_receita_categoria.png'); plt.close(fig)

{1.0: -5.94, 2.0: -9.75, 3.0: -11.07, 4.0: -12.48, 5.0: -13.42}


In [6]:
guf = df.groupby('customer_state')['price'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(guf.index, guf.values, color="#ff7f0e")
ax.set(xlabel="UF", ylabel="receita (R$)", title="Receita por UF (top 10, dados reais)")
fig.tight_layout(); save_fig(fig, 'olist_receita_uf.png'); plt.close(fig)
gf = df.groupby('customer_state')['freight_value'].mean().sort_values(ascending=False).head(10)
print(gf.round(2).to_dict())  # substituto honesto: sem distancia de CD nos dados reais
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(gf.index, gf.values, color="#9467bd")
ax.set(xlabel="UF", ylabel="frete medio (R$)",
       title="Frete medio por UF (top 10) — Norte/Nordeste pagam mais")
fig.tight_layout(); save_fig(fig, 'olist_frete_distancia.png'); plt.close(fig)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].boxplot(items_raw["price"], tick_labels=["price (bruto)"])
axes[0].set_title("Preco antes do clip p99")
axes[1].boxplot(items_raw["freight_value"], tick_labels=["freight (bruto)"])
axes[1].set_title("Frete antes do clip p99")
fig.suptitle("Outliers de preco/frete (bruto -> clip no p99)")
fig.tight_layout(); save_fig(fig, 'olist_outliers_preco_frete.png'); plt.close(fig)

{'RR': 41.78, 'AC': 39.72, 'RO': 39.5, 'PB': 38.32, 'PI': 37.32, 'MA': 36.41, 'SE': 35.29, 'TO': 34.89, 'AL': 34.62, 'RN': 34.62}


## 4. Mineração — KMeans em RFM (k=4), árvore p/ `review_score>=4` + `confusion_summary`, co-ocorrência de categorias

In [7]:
ref = pd.to_datetime(df['order_purchase_timestamp']).max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda s: (ref - pd.to_datetime(s).max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('price', 'sum')).reset_index()
X = StandardScaler().fit_transform(rfm[['recency', 'frequency', 'monetary']])
labels, km = run_kmeans(pd.DataFrame(X), k=4, seed=42)  # KMeans: 4 segmentos
rfm['cluster'] = labels
prof = rfm.groupby('cluster')[['recency', 'frequency', 'monetary']].mean().round(2)
prof['n_clientes'] = rfm.groupby('cluster').size().values
print(prof)
display(HTML(df_to_html_table(prof.reset_index(), max_rows=20)))
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(rfm['recency'], rfm['monetary'], c=rfm['cluster'], s=8, alpha=0.5, cmap='tab10')
ax.set(xlabel="recencia (dias)", ylabel="monetario (R$)",
       title="Segmentos KMeans em RFM (k=4, dados reais)")
fig.colorbar(sc, label='cluster')
fig.tight_layout(); save_fig(fig, 'olist_clusters_rfm.png'); plt.close(fig)

         recency  frequency  monetary  n_clientes
cluster                                          
0         243.50       1.01    696.84        4584
1         393.96       1.00    103.71       37397
2         133.38       1.00    103.14       50558
3         226.16       2.12    242.38        2881


cluster,recency,frequency,monetary,n_clientes
0,243.50,1.01,696.84,4584
1,393.96,1.00,103.71,37397
2,133.38,1.00,103.14,50558
3,226.16,2.12,242.38,2881


In [8]:
feat = df.dropna(subset=['review_score']).copy()
feat['delay_fill'] = feat['delay_days'].fillna(0)
feat['n_items'] = feat.groupby('order_id')['product_id'].transform('count')
Xf = feat[['delay_fill', 'freight_value', 'price', 'freight_ratio', 'n_items', 'is_late']].copy()
Xf['freight_ratio'] = Xf['freight_ratio'].fillna(Xf['freight_ratio'].median())
yf = (feat['review_score'] >= 4).astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.3, random_state=42, stratify=yf)
tree = run_tree(Xtr, ytr)
summ = confusion_summary(yte, tree.predict(Xte))
print(f"accuracy={summ['accuracy']:.4f} confusion_matrix={summ['confusion_matrix']}")
print(pd.Series(tree.feature_importances_, index=Xf.columns).sort_values(ascending=False).round(3).to_dict())
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(summ['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set(xlabel='predito (1=review>=4)', ylabel='real')
ax.set_title(f"Matriz de confusao — arvore (acc={summ['accuracy']:.2f})")
fig.tight_layout(); save_fig(fig, 'olist_matriz_confusao.png'); plt.close(fig)

accuracy=0.7453 confusion_matrix=[[4081, 4129], [4407, 20896]]
{'delay_fill': 0.291, 'freight_ratio': 0.259, 'freight_value': 0.205, 'price': 0.195, 'n_items': 0.05, 'is_late': 0.0}


In [9]:
multi = (df.dropna(subset=['product_category_name'])
             .groupby('order_id')['product_category_name'].apply(lambda s: sorted(set(s))))
pairs = [p for cats in multi if len(cats) > 1 for p in itertools.combinations(cats, 2)]
print(f"pedidos multi-categoria: {sum(1 for cats in multi if len(cats) > 1)}")
cooc = (pd.DataFrame(pairs, columns=['cat_a', 'cat_b']).value_counts()
        .head(10).reset_index(name='n_pedidos'))
display(HTML(df_to_html_table(cooc, max_rows=10)))

pedidos multi-categoria: 727


cat_a,cat_b,n_pedidos
cama_mesa_banho,moveis_decoracao,70
cama_mesa_banho,casa_conforto,43
moveis_decoracao,utilidades_domesticas,24
cama_mesa_banho,utilidades_domesticas,20
bebes,cool_stuff,20
bebes,brinquedos,19
ferramentas_jardim,moveis_decoracao,17
bebes,cama_mesa_banho,17
beleza_saude,esporte_lazer,14
casa_construcao,moveis_decoracao,13


## 5. Amostra publicada — 200 linhas REAIS em `docs/assets/data/sample_olist.csv` (round(2), ≤200KB; leitura verificada com `load_csv`)

In [10]:
sample = (df[['order_id', 'customer_state', 'product_category_name', 'price',
              'freight_value', 'delay_days', 'review_score', 'is_late']]
          .sample(200, random_state=42).sort_values('order_id').round(2))
SAMPLE_CSV.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(SAMPLE_CSV, index=False)
check = load_csv(SAMPLE_CSV)
print(check.shape, check.head(3).to_dict('records'))
import os; print('bytes:', os.path.getsize(SAMPLE_CSV))

(200, 8) [{'order_id': '02b63408e665194f2794a94576001e56', 'customer_state': 'SP', 'product_category_name': 'esporte_lazer', 'price': 189.9, 'freight_value': 49.85, 'delay_days': -10.0, 'review_score': 5.0, 'is_late': 0}, {'order_id': '03d7f508a18d76f89489c40b7008f7e4', 'customer_state': 'RJ', 'product_category_name': 'beleza_saude', 'price': 180.0, 'freight_value': 8.29, 'delay_days': -7.0, 'review_score': 5.0, 'is_late': 0}, {'order_id': '04464191f1b76f60684eac5609f3bdda', 'customer_state': 'SP', 'product_category_name': 'relogios_presentes', 'price': 478.0, 'freight_value': 12.1, 'delay_days': -5.0, 'review_score': 4.0, 'is_late': 0}]
bytes: 15129


## 6. Decisão (se → então, sobre dados reais)
1. Se a nota média despenca de 4,15 (entrega no prazo) para 2,68 (1–7 dias de atraso) e ~1,7 (>7 dias), então o SLA de entrega é a alavanca nº 1 da satisfação (recomenda-se reforçar estoque regional nas UFs de maior receita: SP/RJ/MG).
2. Se o KMeans (k=4) isola 4.584 clientes de alto valor (gasto médio ~R$697) e 2.881 recorrentes (freq. 2,12), então fidelizá-los com frete progressivo (recomenda-se programa de recompra — frete e razão frete/preço somam ~46% da importância na árvore).
3. Se atraso (`delay_fill`, 29% da importância) e razão frete/preço (26%) lideram a árvore (acurácia 0,75) e o frete médio em RR (R$41,78) é ~3× o de SP (R$15,03), então rever a tabela de frete de longa distância (recomenda-se subsidiar/compartilhar frete do Norte/Nordeste).
4. Se `cama_mesa_banho + moveis_decoracao` co-ocorrem em 70 pedidos (top par; 727 pedidos multi-categoria), então criar bundles de casa (recomenda-se kits das categorias associadas).